# Convert outputs to yearly zarr files

In [1]:
import os
import sys
import zarr
import yaml
from glob import glob
from datetime import datetime, timedelta

import numpy as np
import xarray as xr

In [2]:
sys.path.insert(0, os.path.realpath('../libs/'))
import verif_utils as vu

### Get the target data for coord reference

In [3]:
# fn_target = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/C404/C404_GP_2020.zarr'
# ds_target = xr.open_zarr(fn_target)

### Gather diagnostic outputs and merge with prog

In [4]:
year = 1980 #int(args['year'])

source_dir = f'/glade/derecho/scratch/ksha/DWC/RAW_OUTPUT/CONUS_GP_HIST_precip/*{year}*/*{year}*'
fn_all = sorted(glob(source_dir))

ds_collect = []

for fn in fn_all:
    ds = xr.open_dataset(fn)
    ds_collect.append(ds)

ds_final = xr.concat(ds_collect, dim='time')

ds_final = ds_final.rename({'latitude': 'south_north', 'longitude': 'west_east'})
ds_final['west_east'] = np.arange(336).astype(np.float32)
ds_final['south_north'] = np.arange(336).astype(np.float32)
ds_final = ds_final.drop_vars(['forecast_hour'])

load_name = f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_CESM_HIST_{year}.zarr'
ds_prog = xr.open_zarr(load_name)

ds_full = xr.merge([ds_final, ds_prog], join='inner')
ds_full = ds_full.chunk({'time': 12, 'bottom_top': 12, 'south_north': 336, 'west_east': 336})

# ========================================================================== #
# encoding 
dict_encoding = {}
varnames = list(ds_final.keys())
varname_4D = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot_05', 'WRF_P']

chunk_size_3d = dict(chunks=(12, 336, 336))
chunk_size_4d = dict(chunks=(12, 12, 336, 336))
compress = zarr.Blosc(cname='zstd', clevel=1, shuffle=zarr.Blosc.SHUFFLE, blocksize=0)

for i_var, var in enumerate(varnames):
    if var in varname_4D:
        dict_encoding[var] = {'compressor': compress, **chunk_size_4d}
    else:
        dict_encoding[var] = {'compressor': compress, **chunk_size_3d}

save_name = f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/full_output/opt_CESM_HIST_{year}_full.zarr'
ds_full.to_zarr(save_name, mode='w', consolidated=True, compute=True, encoding=dict_encoding)

In [5]:
ds_final

<xarray.Dataset>
Dimensions:         (time: 8759, south_north: 336, west_east: 336)
Coordinates:
  * time            (time) datetime64[ns] 2070-01-01T01:00:00 ... 2070-12-31T...
  * south_north     (south_north) float32 0.0 1.0 2.0 3.0 ... 333.0 334.0 335.0
  * west_east       (west_east) float32 0.0 1.0 2.0 3.0 ... 333.0 334.0 335.0
Data variables:
    WRF_precip_025  (time, south_north, west_east) float32 -0.0008838 ... 0.6926
Attributes:
    Conventions:  CF-1.11

In [6]:
ds_final = ds_final.rename({'latitude': 'south_north', 'longitude': 'west_east'})
ds_final['west_east'] = np.arange(336).astype(np.float32)
ds_final['south_north'] = np.arange(336).astype(np.float32)
ds_final = ds_final.drop_vars(['forecast_hour'])

load_name = f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_B1H_{year}.zarr'
ds_prog = xr.open_zarr(load_name)

ds_full = xr.merge([ds_final, ds_prog])
ds_full = ds_full.chunk({'time': 12, 'bottom_top': 12, 'south_north': 336, 'west_east': 336})

In [7]:
ds_full

<xarray.Dataset>
Dimensions:                  (time: 8784, south_north: 336, west_east: 336,
                              bottom_top: 12)
Coordinates:
  * time                     (time) datetime64[ns] 2024-01-01 ... 2024-12-31T...
  * south_north              (south_north) float32 0.0 1.0 2.0 ... 334.0 335.0
  * west_east                (west_east) float32 0.0 1.0 2.0 ... 334.0 335.0
  * bottom_top               (bottom_top) float32 0.0 1.0 2.0 ... 9.0 10.0 11.0
Data variables: (12/21)
    WRF_precip_025           (time, south_north, west_east) float32 dask.array<chunksize=(12, 336, 336), meta=np.ndarray>
    WRF_radar_composite_025  (time, south_north, west_east) float32 dask.array<chunksize=(12, 336, 336), meta=np.ndarray>
    WRF_OLR                  (time, south_north, west_east) float32 dask.array<chunksize=(12, 336, 336), meta=np.ndarray>
    WRF_TCC                  (time, south_north, west_east) float32 dask.array<chunksize=(12, 336, 336), meta=np.ndarray>
    WRF_GLW                  (time, south_north, west_east) float32 dask.array<chunksize=(12, 336, 336), meta=np.ndarray>
    WRF_SWDOWN               (time, south_north, west_east) float32 dask.array<chunksize=(12, 336, 336), meta=np.ndarray>
    ...                       ...
    WRF_TD2                  (time, south_north, west_east) float32 dask.array<chunksize=(12, 336, 336), meta=np.ndarray>
    WRF_U                    (time, bottom_top, south_north, west_east) float32 dask.array<chunksize=(12, 12, 336, 336), meta=np.ndarray>
    WRF_U10                  (time, south_north, west_east) float32 dask.array<chunksize=(12, 336, 336), meta=np.ndarray>
    WRF_V                    (time, bottom_top, south_north, west_east) float32 dask.array<chunksize=(12, 12, 336, 336), meta=np.ndarray>
    WRF_V10                  (time, south_north, west_east) float32 dask.array<chunksize=(12, 336, 336), meta=np.ndarray>
    forecast_hour            (time) int64 dask.array<chunksize=(12,), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.11

In [10]:
# ========================================================================== #
# encoding 
dict_encoding = {}
varnames = list(ds_final.keys())
varname_4D = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot_05', 'WRF_P']

chunk_size_3d = dict(chunks=(12, 336, 336))
chunk_size_4d = dict(chunks=(12, 12, 336, 336))
compress = zarr.Blosc(cname='zstd', clevel=1, shuffle=zarr.Blosc.SHUFFLE, blocksize=0)

for i_var, var in enumerate(varnames):
    if var in varname_4D:
        dict_encoding[var] = {'compressor': compress, **chunk_size_4d}
    else:
        dict_encoding[var] = {'compressor': compress, **chunk_size_3d}

save_name = f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/full_output/opt_B3H_{year}_full.zarr'
# ds_full.to_zarr(save_name, mode='w', consolidated=True, compute=True, encoding=dict_encoding)

/glade/work/ksha/miniconda3/envs/credit/lib/python3.11/site-packages/xarray/core/concat.py:532: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))
